In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.widgets import Button, Slider
from datetime import datetime
import math

In [ ]:
# function that loads the data, from the csv generated by the 
def readfile(filename):
        # read the CSV file into a dataframe (i.e. like a Python spreadsheet)
    df = pd.read_csv(filename)
    
    # convert dataframe into a 2D Numpy array
    a = df.to_numpy()
    
    # the first column are the counters
    counter_array = a[:,0]
    
    # the second column are the time stamps
    time_array = a[:,1]
    
    # the second column are the time stamps
    # arduino_time_stamp_array = a[:,2]
    
    # the third column are your measurements: either pulseTime or distance depending on how you modified the code above
    temperature_array = a[:,2]
    rh_array = a[:,3]

    return (time_array, temperature_array, rh_array)
    

In [6]:
max_height = 12.3 / 100 # m
min_height = 11 / 100 # m

print(f"Height uncertainty {(max_height - min_height)/ 4}")

Height uncertainty 0.003250000000000003


In [ ]:

# before you copy-paste your constants, note the changes I have made

# https://illingcompany.com/product/12oz-standard-355ml-202x211-brite-can/?srsltid=AfmBOorGi1H67og6nzUoUixZYzyuDW54VpLvHtOzYKGDkJxL2sChPYh_

# Rough dimensions of can
pi = math.pi
r = 3.31 / 100  # eff radius m
h = 11.2 / 100  # height m 

# estimating exposed surface based on cylinder

A = (2 * (pi * (r **2) )) + (h * (2 * pi * r))   # exposed surface in m^3
V = 355 * (1/(100**3)) # volume in m^3

# https://www.engineeringtoolbox.com/standard-atmosphere-d_604.html
rhoair = 1.225 # approx kg/m^3  ROUGH (depends on T)

# https://kg-m3.com/material/aluminum

rhoAl = 2712 # kg/m3  # density Al

# mass of the air is equal to desnity of air * volume of air
mair = rhoair * V # kg air
mAl = 12.46 / 1000 # NOAH REMEMBER TO WEIGH THE CAN TO GET THIS PREISE # mass of can kg
d = 0.095  # rough thickness of Al  

# https://www.engineeringtoolbox.com/thermal-conductivity-metals-d_858.html
k = 237 # conductivity of Al
# print(mair,d)

# https://www.engineeringtoolbox.com/specific-heat-capacity-d_391.html
# go back to make sure this is correct
cair = 1005 # air J/kg/K
cAl = 897 # Al J/kg/K
heatcap = mAl*cAl + mair*cair

Tcel = 273.15 # to convert C to K
Tamb = 23.2 + Tcel # ambient temp (K)

sig = 5.67e-8  # Stefan-Boltzmann W/m^2/K^4

print(f"A: {A}")
print(f"V: {V}")
print(f"mair: {mair}")
print(f"mAl: {mAl}")

In [ ]:
# this function takes eps, kc, start time and start temp as parameters and runs the euler simulation 
# It is called every time these values are changed by a slider
def euler(kc, eps, initalTime, initalTemp, final_time): 
    
    n = 0
    dt = 0.1 # time step (s)
    # If your time step is very low, it will take a long time for the euler simulation to complete, causing the application to stutter. 
    # Try a large-ish time first ~0.1, then go small once you found values you liked. 
    T0   = initalTemp + Tcel   # starting temperature

    tn = initalTime
    Tn = T0

    t = []
    T = []

    while tn < final_time:
        dHc = kc * A * (Tn-Tamb) * dt
        dHr = eps * sig * A * (Tn**4 - Tamb**4) * dt
        Tn = Tn - (dHc + dHr)/heatcap
        tn = tn + dt
        
        delt = (dHc + dHr)*d/k/A/dt  # temp drop across wall of can - check it is tiny
    #    print(delt)
        t.append(tn)     # store the current time
        T.append(Tn-Tcel)     # store the current temperature
        # print(tn,Tn-Tcel)

    return (t, T)

In [ ]:
# # this calculated the chi squared goodness of fit score between two series
# # This is my own herbs and spices, be careful, do not cite this metric, only use it to help you determine what you need

# https://en.wikipedia.org/wiki/Chi-squared_test

# The chi-squared test is asymmetric, so I might have implemented it on the wrong variable, I am not a stats guy
# euler data first, (I didn't want to code up the edge case where you have no values to interpolate)
# Observed is first
def chiSquared(x1s, y1s, x2s, y2s): 
    # since they are not aligned on the time axis, we have to do some interpolation. 
    # (the time axis for the experimental temperatures varies, so even if we matched them, they wouldn't fit)

    # I have chosen to do the interpolation on the euler side, since we can generate more euler points than measured points reducing compute time, but you can always flip it

    sum = 0
    for x1, y1 in zip(x1s, y1s): 
        observed_value = y1 # here euler is first, so we are assuming it is the actual and our data is observed, again might be wrong
        # computer slows when flipped, it seems numpy array ops are slow when doing many thousands of array comparisons, 
        # I don't feel like writing something faster, and this works,

        point_after_index = np.searchsorted(x2s, x1)
        point_before_index = point_after_index - 1


        if point_after_index >= len(y2s):
            # we are at the end of the list, so just use the last interpolation  
            point_after_index -= 1
            point_before_index -= 1

        m = (y2s[point_after_index] - y2s[point_before_index])/ (x2s[point_after_index] - x2s[point_before_index])

        interpolated_value = (m * (x1 - x2s[point_before_index])) + y2s[point_before_index]

        sum += ((observed_value - interpolated_value)**2)/ (interpolated_value)

        # UPDATE: Thursday 20th 11:23. 
        # Checked with Gabe about the chi-2, euler data is observed, so this should be flipped


    return sum

# test case, should be 0.0
a = np.array
assert chiSquared(a([0.5, 1.5, 2.5]), a([1, 3, 5]), a([0, 1, 2, 3]), a([0, 2, 4, 6])) == 0.0
chiSquared(a([0.5, 1.5, 2.5]), a([1, 3, 5]), a([0, 1, 2]), a([0, 2, 4]))


In [ ]:
from os import listdir
from os.path import isfile, join
goodPath = "/Users/sam/Desktop/Science One For Real/physics/SCIE001Physics/dataSorting/bestRuns/"
figures_folder = "/figures"
onlyfiles = [f for f in listdir(goodPath) if isfile(join(goodPath, f))]
onlyfiles

['4.csv', '1.csv', '3.csv', '2.csv']

In [5]:
class Dataset: 
    start = 0
    stop = 0
    kc = 0
    kc_max = 0
    kc_min = 0
    eps = 0
    eps_max = 0
    eps_min = 0
    name = ""

    def __init__(self, start, stop, kc, kc_max, kc_min, eps, eps_max, eps_min, name): 
        self.start = start
        self.stop = stop
        self.kc = kc
        self.kc_max = kc_max
        self.kc_min = kc_min
        self.eps = eps
        self.eps_max = eps_max
        self.eps_min = eps_min
        self.name = name
        


In [ ]:
datasets = [
    Dataset()
]

In [ ]:
for dataset in datasets: 

    time_array, temperature_array, rh_array = readfile(goodPath + dataset.name)

    plt.figure()

    fig, ax = plt.subplots()
    # ax.set_xlabel('Time [s]')

    color = 'tab:blue'
    ax.plot(time_array, temperature_array, marker='o', linestyle='--', color=color, ms=2)
    ax.set_ylabel("Temperature ($^\\circ C$)", color=color)
    # ax.set_ylim([15,50])
    ax.grid(visible=True, axis='x')
    ax.tick_params(axis='y', labelcolor=color)
    ax.set_xlabel('time (s)')

    t, T = euler(init_kc, init_eps, init_time, init_temp, time_array[-1])
    line, = ax.plot(t, T, lw=2, color='tab:red')

    chi_text = ax.text(0.5, 0.5, f'chi-squared {chiSquared(t, T, time_array, temperature_array)}', size=15, transform=ax.transAxes, ha='left', va='top')

    plt.plot(t_array[arrays[1] > 0], arrays[1][arrays[1] > 0], "r-", label="Euler with drag")
    
    plt.plot(t_array[arrays[0] > 0], arrays[0][arrays[0] > 0], "--", label="Euler with drag underestimate")
    plt.plot(t_array[arrays[2] > 0], arrays[2][arrays[2] > 0], "--", label="Euler with drag overestimate")
    
    plt.plot(times, heights, "b.", label="Real data")
    
    plt.title(f"Height vs Time of a Falling Tissue Box With Drag (Trial {idx + 1})")
    plt.xlabel("Time in Seconds")
    plt.ylabel("Height in Meters")
    plt.grid(True)
    plt.legend()
    plt.savefig(figures_folder + f"/EulerFit{idx + 1}.svg", format="svg")
    plt.show()